# 16_Nitrogen_Expansion_Roadmap
## Materia Arche V3.2
### What nitrogen fixation ML actually needs — honest roadmap

The perovskite ML engine is locked (NB15: tau-b 0.2714, CV 0.2889).
This notebook documents what a real nitrogen expansion requires,
without faking results by feeding random numbers into the perovskite model.

In [1]:
import pandas as pd
import numpy as np
from sklearn.ensemble import ExtraTreesRegressor, RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error
from scipy.stats import kendalltau
import warnings
warnings.filterwarnings('ignore')

print("Libraries loaded")

Libraries loaded


## 1. Perovskite ML engine — final status

In [2]:
# Reproduce the locked production model from NB15
df = pd.read_csv("perovskite_stability_clean.csv")

ml_features = ['Perovskite_band_gap', 'Pb', 'Sn', 'I', 'Br', 'Cl',
               'MA', 'FA', 'Cs',
               'first_Prvskt_annealing_temperature', 'first_Prvskt_thermal_annealing_time',
               'Perovskite_thickness', 'Cell_area_measured', 'JV_default_Voc',
               'JV_default_Jsc', 'JV_default_FF']

X = df[ml_features].fillna(0)
y = np.log1p(df['Stability_PCE_T80'])

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Locked model from NB15
et_prod = ExtraTreesRegressor(
    n_estimators=700, max_features='sqrt', min_samples_split=3,
    min_samples_leaf=1, bootstrap=False, random_state=42
)
et_prod.fit(X_train, y_train)
pred = et_prod.predict(X_test)
tau, _ = kendalltau(y_test, pred)
mae = mean_absolute_error(y_test, pred)

print("=" * 65)
print("PEROVSKITE ML ENGINE — FINAL STATUS")
print("=" * 65)
print(f"")
print(f"  Dataset:    {len(df)} devices with T80 stability")
print(f"  Features:   {len(ml_features)} composition + device parameters")
print(f"  Model:      ExtraTreesRegressor (700 trees, sqrt features)")
print(f"  Test tau-b: {tau:.4f}")
print(f"  Test MAE:   {mae:.4f} (log-hours)")
print(f"  CV tau-b:   0.2889 (from NB15 grid search)")
print(f"")
print(f"  Evolution:")
print(f"    NB02: RF baseline          tau-b = 0.2490")
print(f"    NB13: ExtraTrees sweep     tau-b = 0.2675")
print(f"    NB14: Deep ET              tau-b = 0.2706")
print(f"    NB15: Final grid ET        tau-b = {tau:.4f}")
print(f"")
print(f"  Bootstrap: ET better in 81.8% of 1000 resamples")
print(f"  Quantum: 9 experiments, 0 positive lift (closed)")
print(f"")
print(f"  Status: LOCKED for Phase 2")

PEROVSKITE ML ENGINE — FINAL STATUS

  Dataset:    1543 devices with T80 stability
  Features:   16 composition + device parameters
  Model:      ExtraTreesRegressor (700 trees, sqrt features)
  Test tau-b: 0.2714
  Test MAE:   1.2521 (log-hours)
  CV tau-b:   0.2889 (from NB15 grid search)

  Evolution:
    NB02: RF baseline          tau-b = 0.2490
    NB13: ExtraTrees sweep     tau-b = 0.2675
    NB14: Deep ET              tau-b = 0.2706
    NB15: Final grid ET        tau-b = 0.2714

  Bootstrap: ET better in 81.8% of 1000 resamples
  Quantum: 9 experiments, 0 positive lift (closed)

  Status: LOCKED for Phase 2


## 2. What transfers to nitrogen

In [3]:
transfers = pd.DataFrame({
    "Component": [
        "Pipeline architecture",
        "Statistical validation",
        "Milestone-gated decisions",
        "Model selection (NB13-15 sweep)",
        "Evidence packaging",
        "ExtraTrees as default model",
        "Bootstrap CI methodology",
        "Reproducibility framework"
    ],
    "Transfers": ["Yes"] * 8,
    "Notes": [
        "data -> features -> model -> candidates -> validation",
        "Kendall tau-b, 5-fold CV, bootstrap CIs",
        "Go/no-go gates before external spend",
        "RandomizedSearchCV then GridSearchCV refinement",
        "All code, results, and notebooks public",
        "Strong default for tabular materials data",
        "1000-resample paired bootstrap for lift significance",
        "Frozen splits, random_state=42, version-pinned libraries"
    ]
})

does_not_transfer = pd.DataFrame({
    "Component": [
        "Features (composition)",
        "Target variable",
        "Dataset",
        "Trained model weights",
        "Domain knowledge"
    ],
    "Transfers": ["No"] * 5,
    "Why": [
        "Pb/Sn/I/Br/MA/FA/Cs are perovskite-specific. Nitrogen needs d-band center, coordination #, etc.",
        "T80 stability is perovskite-specific. Nitrogen needs binding energy or turnover frequency",
        "Perovskite Database has no nitrogen fixation data",
        "A model trained on perovskite T80 cannot predict nitrogen binding energies",
        "Halide perovskite degradation mechanisms != catalytic N2 reduction"
    ]
})

print("=" * 65)
print("WHAT TRANSFERS TO NITROGEN")
print("=" * 65)
print(transfers.to_string(index=False))
print("")
print("=" * 65)
print("WHAT DOES NOT TRANSFER")
print("=" * 65)
print(does_not_transfer.to_string(index=False))
print("")
print("KEY POINT: The perovskite model CANNOT predict nitrogen results.")
print("Feeding random or made-up numbers into it produces meaningless output.")
print("The methodology transfers. The model and data do not.")

WHAT TRANSFERS TO NITROGEN
                      Component Transfers                                                    Notes
          Pipeline architecture       Yes    data -> features -> model -> candidates -> validation
         Statistical validation       Yes                  Kendall tau-b, 5-fold CV, bootstrap CIs
      Milestone-gated decisions       Yes                     Go/no-go gates before external spend
Model selection (NB13-15 sweep)       Yes          RandomizedSearchCV then GridSearchCV refinement
             Evidence packaging       Yes                  All code, results, and notebooks public
    ExtraTrees as default model       Yes                Strong default for tabular materials data
       Bootstrap CI methodology       Yes     1000-resample paired bootstrap for lift significance
      Reproducibility framework       Yes Frozen splits, random_state=42, version-pinned libraries

WHAT DOES NOT TRANSFER
             Component Transfers                          

## 3. What nitrogen needs — data requirements

In [4]:
requirements = pd.DataFrame({
    "Requirement": [
        "1. Catalyst database",
        "2. Ground truth targets",
        "3. Feature engineering",
        "4. Baseline model",
        "5. Validation protocol"
    ],
    "Description": [
        "Database of N2 fixation catalysts with compositions and performance",
        "DFT-computed N2 binding energies or experimental turnover frequencies",
        "d-band center, coordination number, electronegativity, oxidation state, etc.",
        "ExtraTrees on catalyst features -> binding energy (same pipeline as perovskite)",
        "Kendall tau-b on ranking, CV, bootstrap CIs (same validation as perovskite)"
    ],
    "Source": [
        "Literature scrape or Catalysis-Hub / NOMAD / Materials Project",
        "PySCF/VASP DFT calculations or curated experimental data",
        "Derived from catalyst composition + structure",
        "sklearn (transfers directly from perovskite pipeline)",
        "scipy.stats.kendalltau (transfers directly)"
    ],
    "Effort": [
        "2-4 weeks",
        "4-8 weeks (DFT) or 2-4 weeks (literature)",
        "1-2 weeks",
        "1 week (pipeline exists)",
        "1 day (pipeline exists)"
    ],
    "Status": ["NOT STARTED"] * 5
})

print("=" * 65)
print("NITROGEN DATA REQUIREMENTS")
print("=" * 65)
print(requirements.to_string(index=False))
print("")
print("Total estimated effort: 3-6 months")
print("Blocking dependency: Requirement 1 (catalyst database)")
print("")
print("Potential data sources:")
print("  - Catalysis-Hub (catalysis-hub.org): DFT adsorption energies")
print("  - NOMAD (nomad-lab.eu): computational materials data")
print("  - Materials Project (materialsproject.org): bulk properties")
print("  - Literature scraping: experimental N2 fixation papers")
print("")
print("None of these currently have a ready-to-use N2 fixation dataset")
print("comparable to the Perovskite Database Project (Zenodo 16809654).")

NITROGEN DATA REQUIREMENTS
            Requirement                                                                     Description                                                         Source                                    Effort      Status
   1. Catalyst database             Database of N2 fixation catalysts with compositions and performance Literature scrape or Catalysis-Hub / NOMAD / Materials Project                                 2-4 weeks NOT STARTED
2. Ground truth targets           DFT-computed N2 binding energies or experimental turnover frequencies       PySCF/VASP DFT calculations or curated experimental data 4-8 weeks (DFT) or 2-4 weeks (literature) NOT STARTED
 3. Feature engineering    d-band center, coordination number, electronegativity, oxidation state, etc.                  Derived from catalyst composition + structure                                 1-2 weeks NOT STARTED
      4. Baseline model ExtraTrees on catalyst features -> binding energy (same pipeline 

## 4. Nitrogen milestone framework (template from perovskite)

In [5]:
# Milestone framework for nitrogen — templated from perovskite (NB11)
milestones = pd.DataFrame({
    "Milestone": [
        "N-M1: Dataset assembled",
        "N-M2: ML tau-b > 0.15 vs classical baseline",
        "N-M3: ML p-value < 0.001",
        "N-M4: >= 3 candidates with > 10% predicted improvement",
        "N-M5: Cross-validated stability (5-fold CV)"
    ],
    "Criterion": [
        ">= 200 catalysts with binding energies and compositions",
        "Kendall tau-b on held-out test set",
        "Statistical significance of ranking correlation",
        "Candidates identified by ML that beat classical ranking",
        "CV tau-b within 10% of test tau-b"
    ],
    "Gate": [
        "Required before any ML work",
        "Required before Phase 2 (lab validation)",
        "Required before Phase 2",
        "Required before Phase 2",
        "Required before Phase 2"
    ],
    "Status": ["NOT STARTED"] * 5
})

print("=" * 65)
print("NITROGEN MILESTONE FRAMEWORK")
print("=" * 65)
print(milestones.to_string(index=False))
print("")
print("Note: tau-b threshold is 0.15 (lower than perovskite's 0.20)")
print("because nitrogen fixation is a harder prediction problem with")
print("likely smaller and noisier datasets.")
print("")
print("All milestones must be met before any external lab spend.")
print("This is the same discipline that made perovskite Phase 2 credible.")

milestones.to_csv("Nitrogen_Milestone_Framework.csv", index=False)
print("\nNitrogen_Milestone_Framework.csv exported")

NITROGEN MILESTONE FRAMEWORK
                                             Milestone                                               Criterion                                     Gate      Status
                               N-M1: Dataset assembled >= 200 catalysts with binding energies and compositions              Required before any ML work NOT STARTED
           N-M2: ML tau-b > 0.15 vs classical baseline                      Kendall tau-b on held-out test set Required before Phase 2 (lab validation) NOT STARTED
                              N-M3: ML p-value < 0.001         Statistical significance of ranking correlation                  Required before Phase 2 NOT STARTED
N-M4: >= 3 candidates with > 10% predicted improvement Candidates identified by ML that beat classical ranking                  Required before Phase 2 NOT STARTED
           N-M5: Cross-validated stability (5-fold CV)                       CV tau-b within 10% of test tau-b                  Required before Phase 2

## 5. Updated evidence package

In [6]:
# Updated evidence package — accurate numbers
evidence = pd.DataFrame({
    "Metric": [
        "Dataset", "Best ML model", "Best ML tau-b (test)", "Best ML tau-b (CV)",
        "ML p-value", "ML lift vs RF baseline", "Bootstrap lift > 0",
        "Feature engineering", "Quantum experiments", "Best quantum lift",
        "Production candidates (>20% gain)", "Milestones (ML-focused)",
        "Notebooks", "Nitrogen status"
    ],
    "Value": [
        "1,543 devices with T80 stability",
        "ExtraTreesRegressor (700 trees, sqrt features)",
        "0.2714", "0.2889",
        "< 1e-10", "+0.0224 over RF (n=200)",
        "81.8% of 1000 bootstraps",
        "Did not help (-0.016 lift, NB14)",
        "9 (0 positive lift)", "-0.0024",
        "706", "5/5 met",
        "16 (01-16)", "ON HOLD — methodology proven, data pipeline needed"
    ]
})

print("=" * 65)
print("EVIDENCE PACKAGE — FINAL (accurate)")
print("=" * 65)
print(evidence.to_string(index=False))

evidence.to_csv("Evidence_Package_Final.csv", index=False)
print("\nEvidence_Package_Final.csv exported")

EVIDENCE PACKAGE — FINAL (accurate)
                           Metric                                              Value
                          Dataset                   1,543 devices with T80 stability
                    Best ML model     ExtraTreesRegressor (700 trees, sqrt features)
             Best ML tau-b (test)                                             0.2714
               Best ML tau-b (CV)                                             0.2889
                       ML p-value                                            < 1e-10
           ML lift vs RF baseline                            +0.0224 over RF (n=200)
               Bootstrap lift > 0                           81.8% of 1000 bootstraps
              Feature engineering                   Did not help (-0.016 lift, NB14)
              Quantum experiments                                9 (0 positive lift)
                Best quantum lift                                            -0.0024
Production candidates (>20% g

## 6. Summary

In [7]:
print("=" * 65)
print("SUMMARY — NOTEBOOK 16")
print("=" * 65)
print("")
print("PEROVSKITE:")
print(f"  ML engine LOCKED (ET tau-b {tau:.4f}, CV 0.2889)")
print("  Phase 2: ACTIVE — ready for lab validation")
print("  Next action: Send outreach to Fraunhofer ISE (NB12 templates)")
print("")
print("NITROGEN:")
print("  Status: ON HOLD")
print("  Methodology: PROVEN (transfers from perovskite)")
print("  Data pipeline: NOT STARTED")
print("  Blocking: Need catalyst database (2-4 weeks minimum)")
print("  Milestone framework: DEFINED (5 milestones, 0/5 started)")
print("  Estimated effort: 3-6 months to reach Phase 2 equivalent")
print("")
print("QUANTUM:")
print("  Status: CLOSED as research track")
print("  9 experiments, 0 positive lift")
print("  IBM hardware: accessible but not a gate")
print("")
print("PORTFOLIO:")
print("  Perovskite: Phase 2 ACTIVE (16 notebooks, production model locked)")
print("  Nitrogen: ON HOLD (roadmap defined, data needed)")
print("  Quantum: Separate R&D track (not a gate)")
print("")
print("Notebooks: 16 (01-16)")

SUMMARY — NOTEBOOK 16

PEROVSKITE:
  ML engine LOCKED (ET tau-b 0.2714, CV 0.2889)
  Phase 2: ACTIVE — ready for lab validation
  Next action: Send outreach to Fraunhofer ISE (NB12 templates)

NITROGEN:
  Status: ON HOLD
  Methodology: PROVEN (transfers from perovskite)
  Data pipeline: NOT STARTED
  Blocking: Need catalyst database (2-4 weeks minimum)
  Milestone framework: DEFINED (5 milestones, 0/5 started)
  Estimated effort: 3-6 months to reach Phase 2 equivalent

QUANTUM:
  Status: CLOSED as research track
  9 experiments, 0 positive lift
  IBM hardware: accessible but not a gate

PORTFOLIO:
  Perovskite: Phase 2 ACTIVE (16 notebooks, production model locked)
  Nitrogen: ON HOLD (roadmap defined, data needed)
  Quantum: Separate R&D track (not a gate)

Notebooks: 16 (01-16)
